# Init

In [0]:
# importing liabraries
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import *
from pyspark.sql import Window

# Rename Config

In [0]:
# Rename config
RENAME_MAP = {
    "CID": "cid",
    "CNTRY": "country"
}

# Read from Bronze

In [0]:
# read spark table
df = spark.table("workspace.bronze.erp_loc_a101_raw")

# Transformation

## Rename column

In [0]:
# column Rename
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Extracting customer_key

In [0]:
# Chains both stripping 'NAS' and removing hyphens together
df = df.withColumn(
    "customer_key", 
    F.regexp_replace(F.col("cid"), "-", "")    # Removes hyphens
)



## Trim

In [0]:
# string data triming
for field in df.schema.fields:
    if field.dataType == StringType():
        df = df.withColumn(field.name, trim(col(field.name)))

## Normalizing Country name

In [0]:
# 1. Strip carriage returns (\r), line feeds (\n), and whitespace
clean_country = F.upper(F.trim(F.regexp_replace(F.col("country"), "\r|\n", "")))

# 2. Apply standardisation rules and rename to 'country'
df = df.withColumn(
    "country",
    F.when(clean_country.isin("USA", "UNITED STATES", "US"), "UNITED STATES")
     .when(clean_country.isin("GERMANY", "DE"), "GERMANY")
     .when((clean_country == "") | (clean_country.isNull()), "N/A")
     .otherwise(clean_country)
).drop("cntry") # Drop old column name safely


## Check distinct country

In [0]:
# Check distinct country
display(df.select("country").distinct())


In [0]:
df.display()

# Write in Silver

In [0]:
(
    df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver.erp_loc_a101")
)

In [0]:
%sql
select * from silver.erp_loc_a101